# Annotation Report: Sentiment Analysis (IMDB Movie Reviews)

Strategy: corrected IMDB source labels (inversion fixed) + LLM validation on 20 samples  
Labels: `pos` / `neg`

In [1]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import json, os

df = pd.read_parquet('../data/labeled/labeled.parquet')
with open('../data/annotation/quality_metrics.json') as f:
    metrics = json.load(f)

print(f'Total rows: {len(df)}')
print(f'Labels: {df["auto_label"].value_counts().to_dict()}')
print(f'Confidence mean: {df["confidence"].mean():.3f}')
df.head(3)

Total rows: 4654
Labels: {'neg': 2346, 'pos': 2308}
Confidence mean: 1.000


,text,label,source,collected_at,auto_label,confidence,is_disputed
0,"Ms Aparna Sen, the maker of Mr & Mrs Iyer, dir...",pos,hf:ajaykarthick/imdb-movie-reviews,2026-03-28T04:24:53.802549+00:00,pos,1.0,False
1,"I have seen this film only once, on TV, and it...",pos,hf:ajaykarthick/imdb-movie-reviews,2026-03-28T04:24:53.802549+00:00,pos,1.0,False
2,This marvelous short will hit home with everyo...,pos,hf:ajaykarthick/imdb-movie-reviews,2026-03-28T04:24:53.802549+00:00,pos,1.0,False


In [2]:
# Label distribution
os.makedirs('../data/annotation', exist_ok=True)
dist = df['auto_label'].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
dist.plot(kind='bar', ax=ax1, color=['#e74c3c', '#2ecc71'])
ax1.set_title('Label Distribution')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=0)
for bar, val in zip(ax1.patches, dist.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
             f'{val}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=10)

dist.plot(kind='pie', ax=ax2, autopct='%1.1f%%', colors=['#e74c3c', '#2ecc71'])
ax2.set_title('Label Distribution (pie)')
ax2.set_ylabel('')
plt.tight_layout()
plt.savefig('../data/annotation/label_distribution.png', dpi=120, bbox_inches='tight')
plt.close()
print('Saved label_distribution.png')
print(dist)

Saved label_distribution.png
auto_label
neg    2346
pos    2308
Name: count, dtype: int64


In [3]:
# Confidence distribution
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df['confidence'], bins=20, color='steelblue', edgecolor='white')
ax.axvline(0.7, color='red', linestyle='--', label='Dispute threshold (0.7)')
ax.set_title('Confidence Distribution')
ax.set_xlabel('Confidence')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('../data/annotation/confidence_distribution.png', dpi=120, bbox_inches='tight')
plt.close()
print(f'Disputed (confidence < 0.7): {df["is_disputed"].sum()}')
print(f'Confidence: mean={df["confidence"].mean():.3f}, median={df["confidence"].median():.3f}')

Disputed (confidence < 0.7): 0
Confidence: mean=1.000, median=1.000


In [4]:
# LLM validation on 20-sample results
sample_results = {
    'agreement': '14/20 (70%)',
    'kappa': 0.53,
    'confidence_mean': 0.906,
    'disputed': 0,
    'neutral_found': 3,
    'mixed_found': 3,
}
print('LLM Validation on 20 random samples:')
for k, v in sample_results.items():
    print(f'  {k}: {v}')
print()
print('Key insight: 30% "disagreements" are LLM adding neutral/mixed nuance')
print('that binary pos/neg cannot express — not actual errors.')

LLM Validation on 20 random samples:
  agreement: 14/20 (70%)
  kappa: 0.53
  confidence_mean: 0.906
  disputed: 0
  neutral_found: 3
  mixed_found: 3

Key insight: 30% "disagreements" are LLM adding neutral/mixed nuance
that binary pos/neg cannot express — not actual errors.


## Annotation Approach

### What was done

1. **Label inversion correction** — The dataset `ajaykarthick/imdb-movie-reviews` encodes sentiment as `0=positive, 1=negative`, opposite to standard IMDB convention. Labels were swapped `neg↔pos` to align with the actual text content.

2. **LLM validation on 20 samples** — OpenRouter API (nvidia/nemotron-3-super-120b-a12b) was used to independently classify 20 random reviews with a 4-class taxonomy (pos/neg/neutral/mixed). Agreement = 70%, Cohen's kappa = 0.53 (moderate agreement).

3. **Corrected labels adopted as final** — All 4,654 rows use the corrected IMDB labels (confidence = 1.0). The LLM test confirmed the direction is correct.

### Why the 70% LLM agreement is acceptable

The 30% disagreements are not errors — they reflect genuine label granularity: the LLM correctly identified `neutral` (purely descriptive texts) and `mixed` (contradictory sentiment) cases that binary pos/neg cannot express. For example:
- *"HANDS OF THE RIPPER Aspect ratio: 1.85:1 Sound format: Mono..."* — pure DVD metadata, correctly labeled `neutral` by LLM
- *"The only reason to see this movie is for a brilliant performance..."* — one positive in a sea of negatives, correctly `mixed`

### What was NOT done and why

Full LLM re-annotation of 4,654 rows was skipped (user chose option B). The existing IMDB labels are high quality (human-annotated) once the inversion is corrected. Adding neutral/mixed from LLM on the full set would require ~465 API calls and could introduce LLM noise into an otherwise clean dataset.

In [5]:
# Final summary
print('=== ANNOTATION SUMMARY ===')
print(f'Total rows:         {len(df):,}')
print(f'Labels:             {list(df["auto_label"].unique())}')
print(f'neg:                {(df["auto_label"]=="neg").sum():,} ({(df["auto_label"]=="neg").mean()*100:.1f}%)')
print(f'pos:                {(df["auto_label"]=="pos").sum():,} ({(df["auto_label"]=="pos").mean()*100:.1f}%)')
print(f'Disputed:           0')
print(f'Confidence mean:    1.0')
print(f'LLM validation:     kappa=0.53, agreement=70% on 20 samples')
print(f'Saved:              data/labeled/labeled.parquet')

=== ANNOTATION SUMMARY ===
Total rows:         4,654
Labels:             ['pos', 'neg']
neg:                2,346 (50.4%)
pos:                2,308 (49.6%)
Disputed:           0
Confidence mean:    1.0
LLM validation:     kappa=0.53, agreement=70% on 20 samples
Saved:              data/labeled/labeled.parquet
